# 🎯 Task splitters: drug-target interaction benchmarks

Welcome! The `task` family doesn't implement a generic structural criterion the way `scaffold` or `similarity` do — each class instead encodes a specific downstream deployment question: can the model spot a truly novel **hit**, rank analogues during **lead optimisation**, recognise a **scaffold hop**, generalise to a **cold-start** compound/target/pair, resist the **asymmetric-validation-embedding (AVE)** nearest-neighbour artefact, or survive a property-matched **decoy benchmark**?

Every section below instantiates one class, calls `.split_result(X, y=...)`, and prints a compact summary of the resulting partition sizes plus any splitter-specific metadata.

**Contents**
1. [🔍 HiSplitter — hit identification](#1)
2. [🧪 LoSplitter — lead optimisation](#2)
3. [🦘 ScaffoldHopSplitter — scaffold-hop test](#3)
4. [🥶 Cold-start interaction splitters](#4)
   - [ColdDrugSplitter](#4.1)
   - [ColdTargetSplitter](#4.2)
   - [ColdPairSplitter](#4.3)
5. [⚖️ AVESplitter — asymmetric validation embedding](#5)
6. [🎭 DecoyBenchmarkSplitter — property-matched decoys](#6)

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

import numpy as np

from chemsplit import datasets
from chemsplit.splitters.task import (
    AVESplitter,
    ColdDrugSplitter,
    ColdPairSplitter,
    ColdTargetSplitter,
    DecoyBenchmarkSplitter,
    HiSplitter,
    LoSplitter,
    ScaffoldHopSplitter,
)

# A modest pool of real, distinct small molecules (varied scaffolds).
SMILES = [
    "c1ccccc1", "c1ccccc1C", "c1ccccc1CC", "c1ccncc1", "c1ccncc1C",
    "c1ccc2ccccc2c1", "c1ccc2ccccc2c1C", "C1CCCCC1", "C1CCCCC1C", "C1CCNCC1",
    "c1ccc(cc1)C(=O)O", "c1ccc(cc1)C(=O)OC", "c1ccc(cc1)N", "c1ccc(cc1)NC", "c1ccc(cc1)O",
    "c1ccc(cc1)OC", "CCCCCCCC", "CCCCCCCCC", "c1ccsc1", "c1ccoc1",
    "c1cc2ccccc2[nH]1", "c1cc2ccccc2o1", "C1CCC1", "C1CCC1C", "c1ccc(Cl)cc1",
    "c1ccc(Br)cc1", "c1ccc(F)cc1", "CC(C)C", "CC(C)CC", "CCN",
]


def summarize(name, result):
    print(f"{name}: train={result.train.size} valid={result.valid.size} "
          f"test={result.test.size} discard={result.discard.size} (n={result.n_records})")
    interesting = {k: v for k, v in result.metadata.items()
                   if not isinstance(v, (list, dict)) or len(str(v)) < 80}
    if interesting:
        print("  metadata:", interesting)

<a id="1"></a>
## 1. 🔍 HiSplitter — hit identification

No test molecule may exceed `threshold` similarity to any training molecule — a hard generalisation gap for "can the model spot a truly novel hit?". Sizes are optimised under that hard constraint by one of three deterministic solvers (`greedy`, `ilp`, `annealing`).

| Parameter | Meaning |
|---|---|
| `threshold` | max allowed test-to-train similarity (the guarantee itself) |
| `coarse_cutoff` | coarsening granularity for the underlying conflict-component optimisation |
| `solver` | `'greedy'` / `'ilp'` / `'annealing'` — trade optimality for speed, all deterministic |

> 💡 **Advantages:**
> - Provides a **verified guarantee** — `metadata["max_cross_similarity"]` is checked against the threshold, so the central claim of a hit-identification benchmark is evidence, not assertion.
> - Formulating the assignment over conflict components, rather than pruning greedily, means the guarantee usually costs no data — unlike `similarity_threshold`'s `greedy_prune`.
> - Reveals a genuine, widely reported result: models that look strong under scaffold splitting often fall toward random here. That gap is the finding.
> - Three solvers trade optimality for speed, and all three are deterministic.

> ⚠️ **Pitfalls:**
> - On congeneric or focused datasets the conflict graph collapses into one component and the requested split is simply impossible. Raises rather than silently returning a leaky split.
> - The threshold, fingerprint, and its radius jointly define difficulty — "Tanimoto 0.4" alone isn't a full design.
> - The guarantee is one-sided (test-to-train), so the *train* set may still hold near-duplicates of each other — this controls generalisation, not training redundancy.
> - `verify=False` removes the only proof the split is correct — don't use it for published results.

In [2]:
sp = HiSplitter(threshold=0.3, coarse_cutoff=0.5, train_size=0.7, test_size=0.3, random_state=0)
result = sp.split_result(SMILES)[0]
summarize("HiSplitter", result)

HiSplitter: train=21 valid=0 test=9 discard=0 (n=30)
  metadata: {'realised_sizes': {'train': 21, 'valid': 0, 'test': 9}, 'threshold': 0.3, 'coarse_cutoff': 0.5, 'n_clusters': 26, 'n_components': 9, 'component_sizes': [2, 16, 3, 2, 2, 1, 1, 2, 1], 'max_cross_similarity': 0.30000001192092896, 'n_discarded': 0, 'solver': 'greedy', 'solver_status': 'time_limit_feasible'}


`featurizer`/`metric` (inherited from `SimilarityParamsMixin`, default `"ecfp4"`/`"tanimoto"`) decide what "similarity" means for the hard `threshold` ceiling — switching representation changes which molecules count as too similar to train.

In [3]:
sp_maccs = HiSplitter(threshold=0.6, coarse_cutoff=0.7, featurizer="maccs", metric="dice",
                       train_size=0.7, test_size=0.3, random_state=0)
result_maccs = sp_maccs.split_result(SMILES)[0]
summarize("HiSplitter (maccs/dice)", result_maccs)

HiSplitter (maccs/dice): train=21 valid=0 test=9 discard=0 (n=30)
  metadata: {'realised_sizes': {'train': 21, 'valid': 0, 'test': 9}, 'threshold': 0.6, 'coarse_cutoff': 0.7, 'n_clusters': 14, 'n_components': 8, 'component_sizes': [11, 6, 1, 4, 2, 3, 2, 1], 'max_cross_similarity': 0.6000000238418579, 'n_discarded': 0, 'solver': 'greedy', 'solver_status': 'time_limit_feasible'}


<a id="2"></a>
## 2. 🧪 LoSplitter — lead optimisation

Builds clusters of mutually similar molecules whose activity nevertheless varies (an activity-cliff-rich cluster), then holds one such cluster out whole as the test set — the question a medicinal chemist actually asks: *"which analogue should I make next?"*

| Parameter | Meaning |
|---|---|
| `threshold` | similarity above which molecules can join the same cluster |
| `min_cluster_size` | minimum size for a cluster to qualify as a candidate test cluster |
| `std_threshold` | minimum within-cluster label spread (in the label's own units) to qualify |

> 💡 **Advantages:**
> - Each held-out cluster is internally similar but has real activity spread, so a model that only separates coarse chemotypes scores at chance — invisible to any scaffold or cluster split.
> - `cluster_members` enables the right metric: Spearman correlation **within** each cluster, averaged across clusters, with pooled metrics only as a secondary figure.
> - Train pruning removes the near-neighbour leak that would otherwise make within-cluster ranking trivial.

> ⚠️ **Pitfalls:**
> - **The evaluation metric is part of the split.** Reporting pooled R² or ROC-AUC on a Lo split throws away the whole point — the per-cluster ranking is the result.
> - `std_threshold` is expressed in the label's own units and silently assumes a log scale — nonsense on a linear IC50 column.
> - Train pruning can remove a large fraction of data, and exactly the most informative near-analogues, so absolute scores are incomparable with other splits.
> - Activity spread within a cluster can be assay noise rather than SAR — aggregate replicates and prefer single-assay data.

In [4]:
rng = np.random.default_rng(0)
y = rng.normal(size=len(SMILES))
sp = LoSplitter(threshold=0.3, min_cluster_size=3, std_threshold=0.01, train_size=0.7, test_size=0.3,
                random_state=0)
try:
    result = sp.split_result(SMILES, y=y)[0]
    summarize("LoSplitter", result)
except Exception as exc:
    print(f"LoSplitter: no qualifying cluster on this tiny synthetic pool ({type(exc).__name__})")

LoSplitter: train=11 valid=0 test=17 discard=2 (n=30)
  metadata: {'realised_sizes': {'train': 11, 'valid': 0, 'test': 17}, 'n_clusters': 2, 'cluster_members': [[1, 2, 4, 5, 6, 10, 11, 12, 13, 14, 15, 24, 25, 26], [7, 9, 22]], 'cluster_stds': [0.8911917614675018, 0.9342181495278478], 'cluster_sizes': [14, 3], 'n_pruned_from_train': 2, 'y_std_overall': 0.8091838848815633, 'threshold': 0.3, 'std_threshold': 0.01}


`LoSplitter` also inherits `featurizer`/`metric` — worth checking against the default whenever clusters look unstable, since a different fingerprint can change which molecules cluster together and thus which clusters qualify as low-variance.

In [5]:
sp_lo_physchem = LoSplitter(threshold=0.3, min_cluster_size=3, std_threshold=0.01,
                             featurizer="physchem", metric="cosine",
                             train_size=0.7, test_size=0.3, random_state=0)
try:
    result_lo2 = sp_lo_physchem.split_result(SMILES, y=y)[0]
    summarize("LoSplitter (physchem/cosine)", result_lo2)
except Exception as exc:
    print(f"LoSplitter (physchem/cosine): no qualifying cluster on this tiny synthetic pool ({type(exc).__name__})")

LoSplitter (physchem/cosine): no qualifying cluster on this tiny synthetic pool (EmptyPartitionError)


<a id="3"></a>
## 3. 🦘 ScaffoldHopSplitter — scaffold-hop test

Test-set actives are restricted to scaffolds absent from train, while optionally enforcing a minimum pharmacophoric similarity so the "hop" is still a plausible, recognisable chemical step rather than an unrelated structure.

| Parameter | Meaning |
|---|---|
| `pharmacophore_similarity` | which 2-D pharmacophore proxy to require between test and train actives (`'none'` disables it) |
| `min_pharm_similarity` | minimum pharmacophoric similarity a "hop" must retain |

> 💡 **Advantages:**
> - The correct evaluation for a virtual-screening novelty claim — it demands a new framework while requiring the recognition features stay findable.
> - The scaffold-disjointness constraint is asserted, not assumed.
> - Separating the scaffold criterion from the pharmacophore criterion makes both reportable and tunable.

> ⚠️ **Pitfalls:**
> - Two coupled thresholds (`scaffold_kind`, `min_pharm_similarity`) define the difficulty, neither with a principled default.
> - 2-D pharmacophore similarity is a weak proxy for 3-D recognition — a real hop may fail the criterion, or a pair passing it may bind completely differently.
> - Needs enough actives spread over enough distinct scaffolds; most single-target datasets don't have them, so the splitter raises rather than faking a two-scaffold "benchmark".
> - Inactives dominate screening data's record count, so `inactives_policy` quietly controls the class balance of both partitions.

In [6]:
y = (rng.random(len(SMILES)) > 0.4).astype(np.int64)
sp = ScaffoldHopSplitter(
    pharmacophore_similarity="none", train_size=0.7, test_size=0.3, random_state=0,
    min_pharm_similarity=0.0,
)
try:
    result = sp.split_result(SMILES, y=y)[0]
    summarize("ScaffoldHopSplitter", result)
except Exception as exc:
    print(f"ScaffoldHopSplitter: not enough distinct active scaffolds here ({type(exc).__name__})")

ScaffoldHopSplitter: train=13 valid=0 test=17 discard=0 (n=30)
  metadata: {'realised_sizes': {'train': 13, 'valid': 0, 'test': 17}, 'n_test_scaffolds': 2, 'test_scaffolds': ['c1ccccc1', 'C1CCCCC1'], 'n_active_test': 9, 'n_active_train': 6, 'min_pharm_similarity_achieved': 0.0, 'scaffold_kind': 'generic'}


`pharmacophore_similarity="none"` above turns the pharmacophore-retention criterion **off** — the class default is actually `"fcfp"`. With it on, a train/test scaffold pair must additionally clear `min_pharm_similarity` on a feature-invariant (FCFP) fingerprint, so the hop is scaffold-novel but pharmacophorically recognisable, not just structurally unrelated.

In [7]:
sp_hop_real = ScaffoldHopSplitter(
    pharmacophore_similarity="fcfp", min_pharm_similarity=0.2,
    train_size=0.7, test_size=0.3, random_state=0,
)
try:
    result_hop_real = sp_hop_real.split_result(SMILES, y=y)[0]
    summarize("ScaffoldHopSplitter (fcfp criterion on)", result_hop_real)
except Exception as exc:
    print(f"ScaffoldHopSplitter (fcfp): not enough distinct active scaffolds here ({type(exc).__name__})")

ScaffoldHopSplitter (fcfp criterion on): train=21 valid=0 test=9 discard=0 (n=30)
  metadata: {'realised_sizes': {'train': 21, 'valid': 0, 'test': 9}, 'n_test_scaffolds': 4, 'test_scaffolds': ['C1CCCCC1', 'c1ccc2ccccc2c1', 'C1=CCC=C1', 'C1CCC1'], 'n_active_test': 5, 'n_active_train': 10, 'min_pharm_similarity_achieved': 1.0, 'scaffold_kind': 'generic'}


<a id="4"></a>
## 4. 🥶 Cold-start interaction splitters

`ColdDrugSplitter`, `ColdTargetSplitter`, and `ColdPairSplitter` all build on a compound × target interaction table (see `chemsplit.datasets.make_interactions`) and share one implementation: hold out whole compound groups, whole target groups, or both axes at once, so a held-out interaction never shares its compound (and/or target) with training.

| Parameter | Meaning |
|---|---|
| `compound_grouper` | optional fitted `GroupSplitter` (e.g. `butina`) upgrading "unseen compound" to "unseen chemotype" |
| `target_grouper` | optional fitted `GroupSplitter` (e.g. `sequence_identity`) preventing homologous targets from leaking across the split |
| axis (fixed per class) | `'compound'` / `'target'` / `'pair'` — which entities are held out |

<a id="4.1"></a>
#### ColdDrugSplitter — held-out compounds

> 💡 Matches the most common DTI deployment question — "here is a new compound, which of my known targets does it hit?" — and usually keeps every target represented in training.
>
> ⚠️ The weakest of the three cold-start settings, but routinely reported as if it were the strongest: without `compound_grouper`, held-out compounds are merely *unseen*, not *novel*.

<a id="4.2"></a>
#### ColdTargetSplitter — held-out targets

> 💡 The only cold-start setting supporting a zero-shot-target claim — the main promise of proteochemometrics.
>
> ⚠️ **Homologues leak by default.** Holding out a kinase while training on its 95%-identical paralogue isn't a cold-target experiment — always pair with `target_grouper="sequence_identity"`; the splitter warns every time you don't (see the `HomologyLeakWarning` below).

<a id="4.3"></a>
#### ColdPairSplitter — held out on both axes at once

> 💡 The only setting measuring genuine interaction learning rather than memorised row/column marginals.
>
> ⚠️ Discards the two off-diagonal blocks (typically 50-90% of the data) and its performance is much lower than, and **not comparable to**, `cold_drug`/`cold_target`.

In [8]:
fx = datasets.make_interactions(n_compounds=12, n_targets=6, density=0.4, seed=0)
X = [(fx.smiles[c], fx.targets[t]) for (c, t, _y) in fx.interactions]
y = [yv for (_c, _t, yv) in fx.interactions]
print(f"interaction table: {len(X)} (compound, target) pairs")

result = ColdDrugSplitter(random_state=0).split_result(X, y=y)[0]
summarize("ColdDrugSplitter", result)

result = ColdPairSplitter(random_state=0).split_result(X, y=y)[0]
summarize("ColdPairSplitter", result)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = ColdTargetSplitter(random_state=0).split_result(X, y=y)[0]
summarize("ColdTargetSplitter", result)

interaction table: 25 (compound, target) pairs
ColdDrugSplitter: train=20 valid=0 test=5 discard=0 (n=25)
  metadata: {'realised_sizes': {'train': 20, 'valid': 0, 'test': 5}, 'axis': 'compound', 'n_compounds': 10, 'n_targets': 6, 'n_compound_groups': 10, 'n_target_groups': 6, 'targets_lost_from_train': [], 'compounds_lost_from_train': ['c1ccc2[nH]ccc2c1', 'c1ccc(s1)', 'c1ccc(o1)'], 'records_discarded': 0}
ColdPairSplitter: train=7 valid=0 test=5 discard=13 (n=25)
  metadata: {'realised_sizes': {'train': 7, 'valid': 0, 'test': 5}, 'axis': 'pair', 'n_compounds': 10, 'n_targets': 6, 'n_compound_groups': 10, 'n_target_groups': 6, 'targets_lost_from_train': ['TGT_05', 'TGT_02', 'TGT_01'], 'records_discarded': 13, 'sqrt_fraction_used': 0.4472135954999579, 'discard_frac': 0.52}
ColdTargetSplitter: train=19 valid=0 test=6 discard=0 (n=25)
  metadata: {'realised_sizes': {'train': 19, 'valid': 0, 'test': 6}, 'axis': 'target', 'n_compounds': 10, 'n_targets': 6, 'n_compound_groups': 10, 'n_target_

All three cold-start splitters above pass only `random_state=0`, leaving `compound_grouper`/`target_grouper` at their default `None` — meaning each compound/target is its own singleton group. A held-out "cold" compound is then barely different from a random one: nothing stops a near-duplicate scaffold from sitting on both sides. Passing a real `compound_grouper` (any `GroupSplitter`, e.g. `GenericScaffoldSplitter`) plus `compound_structures` clusters structurally related compounds together first, so a whole scaffold family — not just one compound — is held out.

In [9]:
from chemsplit.splitters.scaffold import GenericScaffoldSplitter

compound_structures = {s: s for s in fx.smiles}
sp_cold_grouped = ColdDrugSplitter(
    compound_grouper=GenericScaffoldSplitter(random_state=0),
    compound_structures=compound_structures,
    random_state=0,
)
result_cold_grouped = sp_cold_grouped.split_result(X, y=y)[0]
summarize("ColdDrugSplitter (scaffold-grouped)", result_cold_grouped)
print("n_compound_groups:", result_cold_grouped.metadata["n_compound_groups"],
      "vs. n_compounds:", result_cold_grouped.metadata["n_compounds"], "(ungrouped == singleton groups)")

ColdDrugSplitter (scaffold-grouped): train=21 valid=0 test=4 discard=0 (n=25)
  metadata: {'realised_sizes': {'train': 21, 'valid': 0, 'test': 4}, 'axis': 'compound', 'n_compounds': 10, 'n_targets': 6, 'n_compound_groups': 5, 'n_target_groups': 6, 'targets_lost_from_train': [], 'compounds_lost_from_train': ['c1ccc(cc1)', 'c1ccc(nc1)'], 'records_discarded': 0}
n_compound_groups: 5 vs. n_compounds: 10 (ungrouped == singleton groups)


<a id="5"></a>
## 5. ⚖️ AVESplitter — asymmetric validation embedding

Minimises the analogue bias that lets a nearest-neighbour classifier "cheat" on a virtual-screening split, via a small genetic-algorithm search over train/test assignments (the `ga` extra).

| Parameter | Meaning |
|---|---|
| `population_size` | GA population size per generation |
| `n_generations` | number of GA generations to run |
| `tolerance` | early-stop once the AVE bias is within this margin of the target |

> 💡 **Advantages:**
> - Removes the specific artefact — actives clustered near actives — that made many published virtual-screening benchmarks trivially solvable by a 1-nearest-neighbour baseline.
> - The bias is one reportable number, computed identically before and after, making the debiasing auditable.
> - Reporting `ave_initial` alone (disabling the GA with a large `tolerance`) is a cheap, valuable audit of any existing benchmark.

> ⚠️ **Pitfalls:**
> - **Aggressive debiasing over-corrects.** Driving AVE to exactly 0 can remove genuine signal along with the artefact — report both `ave_initial` and `ave_final`.
> - AVE depends on the fingerprint and metric, so a "debiased" split is debiased only with respect to that representation.
> - By far the most expensive splitter here, and defined only for binary actives/inactives.
> - Small active counts make AVE unstable, and the GA will happily optimise noise.

In [10]:
y = (rng.random(len(SMILES)) > 0.5).astype(np.int64)
sp = AVESplitter(
    train_size=0.7, test_size=0.3, random_state=0, population_size=6, n_generations=2, tolerance=1.0,
)
result = sp.split_result(SMILES, y=y)[0]
summarize("AVESplitter", result)

AVESplitter: train=21 valid=0 test=9 discard=0 (n=30)
  metadata: {'ave_initial': -0.09300000000000003, 'ave_final': -0.09300000000000003, 'target_bias': 0.0, 'converged': True, 'generations_run': 0, 'auc_AA': 0.307, 'auc_AI': 0.325, 'auc_II': 0.295, 'auc_IA': 0.37, 'n_active_train': 11, 'n_active_test': 5, 'realised_sizes': {'train': 21, 'valid': 0, 'test': 9}}


<a id="6"></a>
## 6. 🎭 DecoyBenchmarkSplitter — property-matched decoys

Builds a property-matched decoy set for a small set of actives (or consumes a curated benchmark's own predefined partition), and groups each active with its decoys so they never straddle the train/test boundary.

| Parameter | Meaning |
|---|---|
| `decoy_ratio` | target number of decoys per active |
| `topology_dissimilarity` | minimum topological dissimilarity required between an active and its decoys |
| `decoy_pool` | the candidate decoy SMILES to draw from |

> 💡 **Advantages:**
> - Property matching removes the trivial signal — actives being heavier, greasier, or more charged than random library molecules.
> - Grouping each active with its own decoys prevents the subtle leak of an active in train and its property twin in test.
> - `mean_active_decoy_similarity` and the shortfall table make the benchmark's construction auditable.

> ⚠️ **Pitfalls:**
> - **Property-matched decoys carry their own hidden bias.** Matching 2-D properties while requiring topological dissimilarity creates a systematic latent difference deep models can learn directly — run `AVESplitter` to measure that residual bias.
> - Decoys are *presumed* inactive, not measured inactive — a few percent are usually real binders, capping achievable precision.
> - `decoy_ratio` fixes the class imbalance and therefore the headline metric — enrichment factors at different ratios aren't comparable.

In [11]:
y = np.zeros(len(SMILES), dtype=np.int64)
y[:6] = 1
decoy_pool = ["CCCCCCCCCC", "CCCCCCCCCCC", "c1ccccc1CCCC", "C1CCCCCC1", "CCOCC", "CCNCC"] * 5
sp = DecoyBenchmarkSplitter(
    decoy_ratio=2, topology_dissimilarity=0.9, decoy_pool=decoy_pool,
    train_size=0.7, test_size=0.3, random_state=0,
)
result = sp.split_result(SMILES, y=y)[0]
summarize("DecoyBenchmarkSplitter", result)

DecoyBenchmarkSplitter: train=21 valid=0 test=9 discard=0 (n=30)
  metadata: {'realised_sizes': {'train': 21, 'valid': 0, 'test': 9}, 'scheme': 'property_matched', 'n_actives': 6, 'n_decoys': 4, 'realised_decoy_ratio': 0.6666666666666666, 'decoy_shortfalls': {0: 2, 3: 2, 4: 2, 5: 2}, 'mean_active_decoy_similarity': 0.0}


That's all 8 `task` splitters. Together they cover the deployment-question spectrum a DTI/virtual-screening model actually faces — from "spot a novel hit" through "generalise to an unseen target" to "survive a property-matched decoy set" — each with its own guarantee, and its own way of being misused.